# NCS embeddings: analysis and model comparison (CPU runtime is fine)

Works on the feature cubes saved by `RGT-Est_NCS_Inference_Colab.ipynb`. No GPU or model is needed. With several cubes (e.g. 2D and 3D) it compares them side by side.

1. **Registration:** NCS coordinates are 0-based sample indices of patch centres. Centres beyond the volume (NCS rounds the patch count up) are dropped, and the rest are mapped exactly onto the seismic grid.
2. **PCA** and **k-means** per cube.
3. **Striping check:** similarity between neighbouring inlines vs neighbouring crosslines. A 2D inline model that's inconsistent from section to section shows much lower similarity across inlines.
4. **Similarity to the well**, and **lateral embedding similarity** compared with the amplitude coherence used by the classical RGT.
5. **SEG-Y outputs** on the seismic grid, per cube.

**What this establishes:** whether the embeddings are registered, consistent, and pick out structure. **What it doesn't:** that clusters are geological facies, or that embeddings improve the LFM or RGT (those need the blind-well and horizon tests).

## 1. Settings

In [ ]:
DATA_SOURCE  = "drive"
DRIVE_FOLDER = "/content/drive/MyDrive/PUBLIC"
DATA_FOLDER  = "/content"
SEGY_FILE    = "Full_Stack_8_41_CROP_1793889427461.sgy"
CUBES        = "auto"            # "auto" = every NCS cube for this seismic in <DRIVE_FOLDER>/outputs, or a list of .zarr names

WELL_IL, WELL_XL = 1123, 2098
WELL_TWT_MS  = 1800
INTERVAL_MS  = (1300, 2300)

N_PCA        = 32
PCA_SAMPLES  = 100_000
L2_NORMALISE = True
K_VALUES     = [4, 6, 8, 12]
K_SHOW       = 8                 # K used in the side-by-side comparison
AMPLITUDE_COHERENCE = True       # also compute the classical RGT's amplitude coherence (~1 min)
SEED         = 0

## 2. Set up

In [ ]:
!pip -q install segyio zarr
import os, glob, time, shutil, sys, json, warnings, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, segyio
warnings.filterwarnings("ignore")
RUN_START = time.time()
if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src, dst = os.path.join(DRIVE_FOLDER, SEGY_FILE), os.path.join(DATA_FOLDER, SEGY_FILE)
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
        shutil.copyfile(src, dst); print("copied", SEGY_FILE)
SEGY_PATH = os.path.join(DATA_FOLDER, SEGY_FILE)
BASE = DRIVE_FOLDER if DATA_SOURCE == "drive" else DATA_FOLDER
stem = os.path.splitext(SEGY_FILE)[0]
!git clone -q https://github.com/fergusrssmith/RGT /content/RGT 2>/dev/null || git -C /content/RGT pull -q
sys.path[:0] = ["/content/RGT/colab", "/content/RGT/demo"]
import rgt_lfm as R, ncs_tools as N
for m in (R, N): importlib.reload(m)
OUT_DIR = os.path.join(BASE, "outputs", time.strftime("%Y-%m-%d_%H%M", time.gmtime(RUN_START)) + "_ncs_analysis")
os.makedirs(OUT_DIR, exist_ok=True)

G = R.read_segy_cube(SEGY_PATH)
cube, ilines, xlines, samples, dt = G["cube"], G["ilines"], G["xlines"], G["samples"], G["dt"]
NI, NX, NT = cube.shape
ci = int(np.searchsorted(ilines, WELL_IL)) if WELL_IL in ilines else NI // 2
cx = int(np.searchsorted(xlines, WELL_XL)) if WELL_XL in xlines else NX // 2
k_t = int(np.clip(round((sum(INTERVAL_MS) / 2 - samples[0]) / dt), 0, NT - 1))
vclip = np.percentile(np.abs(cube), 99)
EXT_IL = (xlines[0], xlines[-1], samples[-1], samples[0]); EXT_MAP = (xlines[0], xlines[-1], ilines[0], ilines[-1])
print(f"seismic: {NI} × {NX} × {NT}; header bytes {G['il_byte']}/{G['xl_byte']}")

## 3. Find the feature cubes and check registration

In [ ]:
paths = sorted(glob.glob(os.path.join(BASE, "outputs", f"{stem}_ncs_*.zarr"))) if CUBES == "auto" else \
        [os.path.join(BASE, "outputs", c) for c in CUBES]
if not paths:
    raise FileNotFoundError("No NCS feature cubes found. Run RGT-Est_NCS_Inference_Colab.ipynb first.")
CUBE = {}
for p in paths:
    name = os.path.basename(p)[len(stem) + 5:-5] or "ncs"            # e.g. "3d_dir0_dens4", or "dir0_dens4" (first 2D run)
    if not name.startswith(("2d", "25d", "3d")): name = "2d_" + name
    ds, grp = N.open_cube(p)
    ds_c, dropped = N.crop_to_volume(ds, NI, NX, NT)
    rec = json.load(open(p + ".json")) if os.path.exists(p + ".json") else {}
    print(f"=== {name}  (group {grp}; {dict(ds.sizes)})" + (f"  [{rec.get('gpu')}, {rec.get('minutes')} min]" if rec else ""))
    print(f"   centres outside the volume dropped: {dropped}")
    print("   " + N.describe_registration(ds_c, ilines, xlines, samples).replace("\n", "\n   "))
    MAPS = tuple(N.axis_maps(ds_c[a].values, n) for a, n in (("inline", NI), ("xline", NX), ("time_depth", NT)))
    CUBE[name] = dict(ds=ds_c, F=ds_c["features"], MAPS=MAPS)

## 4. PCA, k-means, similarity to the well, lateral similarity (per cube)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
k_ref = int(np.clip(round((WELL_TWT_MS - samples[0]) / dt), 0, NT - 1))
rows = []
for name, c in CUBE.items():
    t0 = time.time(); F, MAPS = c["F"], c["MAPS"]
    Xs = N.sample_vectors(F, PCA_SAMPLES, seed=SEED, normalise=L2_NORMALISE)
    c["pca"] = PCA(n_components=N_PCA, svd_solver="randomized", random_state=SEED).fit(Xs)
    c["P"] = N.pca_transform(F, c["pca"], normalise=L2_NORMALISE)
    flatP = c["P"].reshape(-1, N_PCA)
    fit_idx = np.random.default_rng(SEED).choice(len(flatP), min(250_000, len(flatP)), replace=False)
    c["LAB"] = {K: MiniBatchKMeans(n_clusters=K, random_state=SEED, n_init=5, batch_size=10_000)
                   .fit(flatP[fit_idx]).predict(flatP).reshape(c["P"].shape[:3]).astype(np.int16) for K in K_VALUES}
    fi, fx, ft = (int(MAPS[0][1][ci]), int(MAPS[1][1][cx]), int(MAPS[2][1][k_ref]))
    c["SIM"] = N.cosine_to(F, np.asarray(F.isel(inline=fi, xline=fx, time_depth=ft).values, np.float32))
    s_il, s_xl = N.lateral_similarity(F)
    c["LATSIM"] = N.node_from_links(s_il, s_xl)
    a_il, a_xl = N.striping(s_il, s_xl)
    cum = np.cumsum(c["pca"].explained_variance_ratio_)
    rows.append(dict(cube=name, grid=tuple(c["P"].shape[:3]), var_8PC=cum[min(7, N_PCA - 1)] * 100, var_32PC=cum[-1] * 100,
                     sim_across_inlines=a_il, sim_across_crosslines=a_xl, ratio_il_to_xl=a_il / a_xl,
                     seconds=round(time.time() - t0)))
summary = pd.DataFrame(rows)
print(summary.round(3).to_string(index=False))
print("\nratio_il_to_xl well below 1 for a 2D inline model = sections embedded inconsistently (striping); ~1 = consistent.")

## 5. Side-by-side comparison

Each row is one cube. Columns: PC1–3 on the inline through the well, PC1–3 on a time slice, k-means (`K_SHOW`) on the time slice, and lateral similarity on the time slice.
**On the time slices, look for stripes parallel to the inlines.** They show the model's section-by-section inconsistency, not geology.

In [ ]:
def rgb3(P, MAPS, idx):
    rgb = np.stack([N.to_seismic_grid(P[..., k], MAPS)[idx] for k in range(3)], -1)
    lo, hi = np.percentile(rgb, [2, 98], axis=tuple(range(rgb.ndim - 1)))
    return np.clip((rgb - lo) / (hi - lo + 1e-9), 0, 1)
n = len(CUBE)
fig, axes = plt.subplots(n, 4, figsize=(24, 5.2 * n), squeeze=False, gridspec_kw={"width_ratios": [2, 1.2, 1.2, 1.2]})
for r, (name, c) in enumerate(CUBE.items()):
    axes[r, 0].imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-vclip, vmax=vclip, extent=EXT_IL)
    axes[r, 0].imshow(rgb3(c["P"], c["MAPS"], (ci, slice(None), slice(None))).transpose(1, 0, 2), aspect="auto", extent=EXT_IL, alpha=0.6)
    axes[r, 0].set_title(f"{name}: PC1–3, inline {ilines[ci]}"); axes[r, 0].set_ylabel("TWT (ms)")
    axes[r, 1].imshow(rgb3(c["P"], c["MAPS"], (slice(None), slice(None), k_t)), origin="lower", aspect="auto", extent=EXT_MAP)
    axes[r, 1].set_title(f"{name}: PC1–3 at {samples[k_t]:.0f} ms")
    lab = N.to_seismic_grid(c["LAB"][K_SHOW], c["MAPS"], order=0)
    axes[r, 2].imshow(lab[:, :, k_t], cmap="tab20", vmin=0, vmax=19, origin="lower", aspect="auto", extent=EXT_MAP, interpolation="nearest")
    axes[r, 2].set_title(f"{name}: K={K_SHOW} at {samples[k_t]:.0f} ms")
    ls = N.to_seismic_grid(c["LATSIM"], c["MAPS"])[:, :, k_t]
    axes[r, 3].imshow(ls, cmap="gray", origin="lower", aspect="auto", extent=EXT_MAP, vmin=np.percentile(ls, 2), vmax=np.percentile(ls, 99.5))
    axes[r, 3].set_title(f"{name}: lateral similarity at {samples[k_t]:.0f} ms")
    for ax in axes[r, 1:]: ax.plot(xlines[cx], ilines[ci], "r*", ms=10); ax.set_ylabel("inline")
fig.tight_layout(); plt.show()

## 6. K-means sections for each K (per cube)

In [ ]:
for name, c in CUBE.items():
    fig, axes = plt.subplots(1, len(K_VALUES), figsize=(5.5 * len(K_VALUES), 5))
    for ax, K in zip(np.atleast_1d(axes), K_VALUES):
        lab = N.to_seismic_grid(c["LAB"][K], c["MAPS"], order=0)
        ax.imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-vclip, vmax=vclip, extent=EXT_IL)
        ax.imshow(lab[ci].T, cmap="tab20", aspect="auto", extent=EXT_IL, alpha=0.5, vmin=0, vmax=19, interpolation="nearest")
        ax.set_title(f"{name}: K={K}, inline {ilines[ci]}")
    fig.tight_layout(); plt.show()

## 7. Similarity to the well, and lateral similarity vs amplitude coherence

- **Similarity to the well:** cosine similarity to the embedding at `WELL_IL`, `WELL_XL`, `WELL_TWT_MS`. The candidate weighting for facies-aware LFM interpolation.
- **Lateral similarity vs amplitude coherence:** if an embedding version highlights faults and the chaotic zone below Ranger more clearly, it's a candidate signal for the classical/hybrid RGT.

In [ ]:
COH = None
if AMPLITUDE_COHERENCE:
    import rgt_classic as C; importlib.reload(C)
    t0 = time.time(); _p, _q, _qual, LINKS = C.fault_aware_dips(cube); COH = C.voxel_weight_from_links(LINKS, power=1)
    print(f"amplitude coherence: {time.time()-t0:.0f} s")
n = len(CUBE) + (COH is not None)
fig, axes = plt.subplots(n, 2, figsize=(20, 5.2 * n), squeeze=False, gridspec_kw={"width_ratios": [2, 1.2]})
r = 0
if COH is not None:
    axes[0, 0].imshow(COH[ci].T, cmap="gray", aspect="auto", extent=EXT_IL, vmin=np.percentile(COH, 2), vmax=1)
    axes[0, 0].set_title(f"amplitude coherence (classical RGT), inline {ilines[ci]}")
    axes[0, 1].imshow(COH[:, :, k_t], cmap="gray", origin="lower", aspect="auto", extent=EXT_MAP, vmin=np.percentile(COH, 2), vmax=1)
    axes[0, 1].set_title(f"amplitude coherence at {samples[k_t]:.0f} ms"); r = 1
for name, c in CUBE.items():
    sim = N.to_seismic_grid(c["SIM"], c["MAPS"])
    axes[r, 0].imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-vclip, vmax=vclip, extent=EXT_IL)
    im = axes[r, 0].imshow(sim[ci].T, cmap="magma", aspect="auto", extent=EXT_IL, alpha=0.6)
    axes[r, 0].plot(xlines[cx], samples[k_ref], "c*", ms=14); axes[r, 0].set_title(f"{name}: similarity to the well embedding")
    plt.colorbar(im, ax=axes[r, 0])
    ls = N.to_seismic_grid(c["LATSIM"], c["MAPS"])
    axes[r, 1].imshow(ls[:, :, k_t], cmap="gray", origin="lower", aspect="auto", extent=EXT_MAP,
                      vmin=np.percentile(ls, 2), vmax=np.percentile(ls, 99.5))
    axes[r, 1].set_title(f"{name}: lateral similarity at {samples[k_t]:.0f} ms")
    if COH is not None:
        c["r_coh"] = np.corrcoef(COH[:, :, ::4].ravel(), ls[:, :, ::4].ravel())[0, 1]
    r += 1
fig.tight_layout(); plt.show()
if COH is not None:
    print("correlation of lateral similarity with amplitude coherence:", {k: round(float(c["r_coh"]), 2) for k, c in CUBE.items()})

## 8. Save on the seismic grid (SEG-Y, headers copied from the input)

In [ ]:
def write_like_input(volume, tag):
    path = os.path.join(OUT_DIR, f"{stem}_{tag}.sgy")
    shutil.copyfile(SEGY_PATH, path)
    with segyio.open(path, "r+", ignore_geometry=True) as f:
        for k in range(G["n_tr"]):
            f.trace[k] = np.ascontiguousarray(volume[G["il_idx"][k], G["xl_idx"][k]], dtype=np.float32)
    print("wrote", os.path.basename(path))
for name, c in CUBE.items():
    for K in K_VALUES:
        write_like_input(N.to_seismic_grid(c["LAB"][K], c["MAPS"], order=0).astype(np.float32), f"ncs_{name}_facies_K{K}")
    for k in range(3):
        write_like_input(N.to_seismic_grid(c["P"][..., k], c["MAPS"]), f"ncs_{name}_PC{k+1}")
    write_like_input(N.to_seismic_grid(c["SIM"], c["MAPS"]), f"ncs_{name}_similarity_to_well_{WELL_TWT_MS}ms")
    write_like_input(N.to_seismic_grid(c["LATSIM"], c["MAPS"]), f"ncs_{name}_lateral_similarity")
    np.save(os.path.join(OUT_DIR, f"{stem}_ncs_{name}_pca{N_PCA}_featuregrid.npy"), c["P"].astype(np.float16))
summary.to_csv(os.path.join(OUT_DIR, f"{stem}_ncs_summary.csv"), index=False)
print("all outputs in", OUT_DIR)